# TemporalRec — Iterative User-and-Item 5-Core Filter

This notebook applies the standard sequential-recommendation **5-core** rule to our cleaned Retailrocket events.

- A visitor must have **at least 5** eligible interactions.
- An item must have **at least 5** eligible interactions across all retained visitors.
- Removing rare items can make a visitor rare, so we repeat both checks until the data is stable.

The raw file and prior cleaned files are never changed. This notebook writes a new versioned output folder.

## Before you run

Run Notebook 01 first. We already chose all three event types as the core event policy, so this notebook reads its tie-free output. Update `INPUT_RUN_TAG` if your Notebook 01 run used a different value.

If you rerun this notebook, change `FILTER_RUN_TAG`; it will refuse to overwrite an older audit.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

pd.set_option("display.max_columns", 30)

PROJECT_ROOT = Path.cwd()
INPUT_RUN_TAG = "notebook_run_2026_10_13"  # Must match Notebook 01.
FILTER_RUN_TAG = "five_core_2026_10_13"
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

INPUT_FILE = PROJECT_ROOT / "data" / "interim" / INPUT_RUN_TAG / "events_clean_for_next_steps.csv"
REPORT_DIR = PROJECT_ROOT / "reports" / FILTER_RUN_TAG
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / FILTER_RUN_TAG

assert INPUT_FILE.exists(), f"Cannot find {INPUT_FILE}. Run Notebook 01 first or correct INPUT_RUN_TAG."
assert not REPORT_DIR.exists(), f"{REPORT_DIR} exists. Change FILTER_RUN_TAG rather than overwriting an old audit."
REPORT_DIR.mkdir(parents=True)
INTERIM_DIR.mkdir(parents=True)

print(f"Input:   {INPUT_FILE}")
print(f"Reports: {REPORT_DIR}")
print(f"Output:  {INTERIM_DIR}")

In [ ]:
EVENT_COLUMNS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
DTYPES = {
    "timestamp": "int64",
    "visitorid": "int64",
    "event": "category",
    "itemid": "int64",
    "transactionid": "Int64",
}
events = pd.read_csv(INPUT_FILE, dtype=DTYPES)
assert events.columns.tolist() == EVENT_COLUMNS
assert events.duplicated(EVENT_COLUMNS, keep=False).sum() == 0
assert events.duplicated(["visitorid", "timestamp"], keep=False).sum() == 0

run_metadata = pd.DataFrame([
    {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "input_file": str(INPUT_FILE.relative_to(PROJECT_ROOT)),
        "input_rows": len(events),
        "min_user_interactions": MIN_USER_INTERACTIONS,
        "min_item_interactions": MIN_ITEM_INTERACTIONS,
    }
])
run_metadata.to_csv(REPORT_DIR / "00_filter_metadata.csv", index=False)
print(f"Input events: {len(events):,}")

## Apply the filter until it is stable

Each loop has two parts: remove rare visitors, then remove rare items. The final loop must remove zero rows; that proves the data is stable under both rules.

In [ ]:
current = events.copy()
iteration_rows = []
iteration = 1

while True:
    input_rows = len(current)
    input_users = current["visitorid"].nunique()
    input_items = current["itemid"].nunique()

    user_counts = current.groupby("visitorid").size()
    valid_users = user_counts[user_counts >= MIN_USER_INTERACTIONS].index
    after_user_filter = current.loc[current["visitorid"].isin(valid_users)].copy()

    item_counts = after_user_filter.groupby("itemid").size()
    valid_items = item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index
    after_item_filter = after_user_filter.loc[after_user_filter["itemid"].isin(valid_items)].copy()

    rows_removed_by_user_rule = input_rows - len(after_user_filter)
    rows_removed_by_item_rule = len(after_user_filter) - len(after_item_filter)
    total_rows_removed = input_rows - len(after_item_filter)

    iteration_rows.append({
        "iteration": iteration,
        "input_rows": input_rows,
        "input_users": input_users,
        "input_items": input_items,
        "users_below_minimum": int((user_counts < MIN_USER_INTERACTIONS).sum()),
        "rows_removed_by_user_rule": rows_removed_by_user_rule,
        "items_below_minimum_after_user_rule": int((item_counts < MIN_ITEM_INTERACTIONS).sum()),
        "rows_removed_by_item_rule": rows_removed_by_item_rule,
        "output_rows": len(after_item_filter),
        "output_users": after_item_filter["visitorid"].nunique(),
        "output_items": after_item_filter["itemid"].nunique(),
        "total_rows_removed_this_iteration": total_rows_removed,
    })

    if total_rows_removed == 0:
        break

    current = after_item_filter
    iteration += 1

five_core_events = after_item_filter.copy()
iteration_summary = pd.DataFrame(iteration_rows)
iteration_summary.to_csv(REPORT_DIR / "01_iteration_summary.csv", index=False)
iteration_summary

## Verify the final data and save review files

The user and item count files are intentionally small enough for Excel. Every `interaction_count` must be at least five.

In [ ]:
final_user_counts = (
    five_core_events.groupby("visitorid").size()
    .rename("interaction_count")
    .reset_index()
    .sort_values(["interaction_count", "visitorid"], kind="stable")
)
final_item_counts = (
    five_core_events.groupby("itemid").size()
    .rename("interaction_count")
    .reset_index()
    .sort_values(["interaction_count", "itemid"], kind="stable")
)

assert (final_user_counts["interaction_count"] >= MIN_USER_INTERACTIONS).all()
assert (final_item_counts["interaction_count"] >= MIN_ITEM_INTERACTIONS).all()
assert five_core_events.duplicated(EVENT_COLUMNS, keep=False).sum() == 0
assert five_core_events.duplicated(["visitorid", "timestamp"], keep=False).sum() == 0

five_core_events.to_csv(INTERIM_DIR / "events_all_event_types_5core.csv", index=False)
final_user_counts.to_csv(REPORT_DIR / "02_final_user_counts.csv", index=False)
final_item_counts.to_csv(REPORT_DIR / "03_final_item_counts.csv", index=False)

final_verification = pd.DataFrame([
    {"check": "final events", "value": len(five_core_events)},
    {"check": "final visitors", "value": five_core_events["visitorid"].nunique()},
    {"check": "final items", "value": five_core_events["itemid"].nunique()},
    {"check": "minimum final visitor interaction count", "value": final_user_counts["interaction_count"].min()},
    {"check": "minimum final item interaction count", "value": final_item_counts["interaction_count"].min()},
    {"check": "exact duplicate members remaining", "value": int(five_core_events.duplicated(EVENT_COLUMNS, keep=False).sum())},
    {"check": "timestamp-tie members remaining", "value": int(five_core_events.duplicated(["visitorid", "timestamp"], keep=False).sum())},
])
final_verification.to_csv(REPORT_DIR / "04_final_verification.csv", index=False)
final_verification

## Manual Excel verification

Open the CSV files in `reports/<FILTER_RUN_TAG>/`.

1. In `01_iteration_summary.csv`, the final row must have `total_rows_removed_this_iteration = 0`. That means the loop became stable.
2. In `02_final_user_counts.csv`, sort `interaction_count` smallest to largest. The first value must be 5 or more.
3. Repeat for `03_final_item_counts.csv`. Its first count must also be 5 or more.
4. In `04_final_verification.csv`, duplicate and timestamp-tie counts must be zero.

The next step, after you review these outputs, is chronological train/validation/test splitting and creating next-item labels.